# 06. Ablation Diagnostics

本 notebook 只读取 03–05 的实际 artifacts。Synthetic recovery、coverage/SBC、posterior checks 和 held-out PPC 只有在训练真实发生后才会被计算；否则明确标记 `Insufficient evidence`。

新增维度的价值只能定义为更好的 recovery 或 held-out PPC，同时保持 calibration 和 robustness。更窄 posterior 不是质量证据。

In [1]:
from __future__ import annotations
import json
import os
from pathlib import Path
import sys
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / ".git").exists() and (PROJECT_ROOT / "S4_sbi").exists()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate repository root")
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "S4_sbi" / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from sleep_sbi import overnight_ablation as oa

print("project root:", PROJECT_ROOT)
print("CONDA_DEFAULT_ENV:", os.environ.get("CONDA_DEFAULT_ENV"))
print("sys.executable:", sys.executable)
print("sys.prefix:", sys.prefix)
print("Python:", sys.version)
print("workflow version:", oa.SCHEMA_VERSION)
assert os.environ.get("CONDA_DEFAULT_ENV") == "neurolib"
assert "neurolib" in sys.executable.lower()
assert "neurolib" in sys.prefix.lower()

project root: D:\Year3_Mao_Projects\sleep_loop
CONDA_DEFAULT_ENV: neurolib
sys.executable: C:\Users\YUS190\AppData\Local\anaconda3\envs\neurolib\python.exe
sys.prefix: C:\Users\YUS190\AppData\Local\anaconda3\envs\neurolib
Python: 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:36:49) [MSC v.1944 64 bit (AMD64)]
workflow version: overnight-observation-ablation-v0.1


## Read actual ablation state

不手工填写任何 posterior、coverage 或 PPC 结果。

In [2]:
diagnostic_result = oa.run_ablation_diagnostics()
display(diagnostic_result["summary"])
print(json.dumps(diagnostic_result["report"], indent=2))
assert diagnostic_result["report"]["training_started"] is False

,schema,dimension,training_status,train_simulations,seeds_completed,runtime_s,recovery,coverage,held_out_ppc,failures,interpretation
0,A_baseline14,14,Blocked by parity,0,0,0.0,Insufficient evidence: training not run,Insufficient evidence: training not run,Insufficient evidence: training not run,Blocked by extractor parity,Do not compare posterior width or select a sch...
1,B_baseline_plus_so_morphology,16,Blocked by parity,0,0,0.0,Insufficient evidence: training not run,Insufficient evidence: training not run,Insufficient evidence: training not run,Blocked by extractor parity,Do not compare posterior width or select a sch...
2,C_baseline_plus_spindle_occupancy,15,Blocked by parity,0,0,0.0,Insufficient evidence: training not run,Insufficient evidence: training not run,Insufficient evidence: training not run,Blocked by extractor parity,Do not compare posterior width or select a sch...
3,D_baseline_plus_event_phase,17,Blocked by parity,0,0,0.0,Insufficient evidence: training not run,Insufficient evidence: training not run,Insufficient evidence: training not run,Blocked by extractor parity,Do not compare posterior width or select a sch...
4,E_recommended_frozen_augmented,not frozen,Blocked by parity,0,0,0.0,Insufficient evidence: training not run,Insufficient evidence: training not run,Insufficient evidence: training not run,Blocked by extractor parity,Do not compare posterior width or select a sch...
5,F_complete23_diagnostic,23,Blocked by parity,0,0,0.0,Insufficient evidence: training not run,Insufficient evidence: training not run,Insufficient evidence: training not run,Blocked by extractor parity,Do not compare posterior width or select a sch...


{
  "schema_version": "overnight-observation-ablation-v0.1",
  "created_utc": "2026-07-27T01:46:45.334051+00:00",
  "training_started": false,
  "synthetic_recovery": "not run",
  "coverage_calibration": "not run",
  "held_out_ppc": "not run",
  "posterior_quality": "not run",
  "conclusion": "No scheme can be selected for SNPE before extractor semantic parity is established. Narrower posterior would not constitute evidence even if one were produced.",
  "minimum_next_step": "Build and validate an observable-level simulator adapter, then repeat schema freeze and parity before training."
}


## End-to-end reload and source protection

重新读取 JSON/CSV/NPZ，拒绝 object arrays，并在最终状态重算五个受保护 notebooks 的 SHA-256。

In [3]:
reload_report = oa.reload_overnight_artifacts()
display(pd.DataFrame(reload_report["checks"]))
print("all artifact reload checks pass:", reload_report["all_pass"])
print("source notebook hashes at end:")
for path, digest in reload_report["source_notebook_sha256_end"].items():
    print(path, digest)
print("git status --short at end:")
print(reload_report["git_status_short_end"])
print("git diff --stat at end:")
print(reload_report["git_diff_stat_end"])
assert reload_report["all_pass"]

,path,kind,ok,rows,object_arrays,keys
0,S4_sbi/results/overnight_observation_ablation/...,json,True,NaN,NaN,NaN
1,S4_sbi/results/overnight_observation_ablation/...,json,True,NaN,NaN,NaN
2,S4_sbi/results/overnight_observation_ablation/...,csv,True,23.0,NaN,NaN
3,S4_sbi/results/overnight_observation_ablation/...,csv,True,6.0,NaN,NaN
4,S4_sbi/results/overnight_observation_ablation/...,csv,True,85.0,NaN,NaN
5,S4_sbi/results/overnight_observation_ablation/...,json,True,NaN,NaN,NaN
6,S4_sbi/results/overnight_observation_ablation/...,json,True,NaN,NaN,NaN
7,S4_sbi/results/overnight_observation_ablation/...,npz,True,NaN,[],"[approved_schema_names, train_indices, validat..."
8,S4_sbi/results/overnight_observation_ablation/...,csv,True,6.0,NaN,NaN
9,S4_sbi/results/overnight_observation_ablation/...,csv,True,6.0,NaN,NaN


all artifact reload checks pass: True
source notebook hashes at end:
S4_sbi/notebooks/01_observation.ipynb 10045788079b7c65b55820054c0e7937f5f9adb8b736fb90ebb056a38101d755
S4_sbi/notebooks/01_observation_annotated.ipynb 91c349078bc1ed5c0f53f4b906b9dbf02e8c28422e010d7c568f79bbd27694eb
S4_sbi/notebooks/01_Pyloric_Inspired_Observation.ipynb b954eb278536b8042da2a4651724e66387377cbd89fd22f70245aa8bd1db8c3c
S4_sbi/notebooks/01_Pyloric_Inspired_Observation_annotated.ipynb 7e1ad5b64ce2b93fba765939a5eca86d665ab0776ceb886a23bf0b7105dd6a8f
S4_sbi/notebooks/02_Observation_Schema_Comparison.ipynb ccecfdcd857967112d194837b0f3e993f4c96ecfe01b1394d39da4898b7589b3
git status --short at end:
 D S4_sbi/compute_xobs_from_eeg.py
 D S4_sbi/compute_xobs_from_eeg_v1_buggy.py
 D S4_sbi/compute_xobs_from_eeg_v2.py
 M S4_sbi/compute_xobs_from_eeg_v4.py
 M S4_sbi/notebooks/01_Pyloric_Inspired_Observation_annotated.ipynb
 D S4_sbi/plot_scan_diagnostics.py
 D S4_sbi/run_sbc_standalone.py
 D S4_sbi/scan_xobs_params.

## Conclusion

当前没有 synthetic recovery、coverage、SBC、L-C2ST 或 posterior predictive conclusions，因为没有合格的 simulator-to-EEG observation parity。下一步是实现并验证 observable-level adapter，再重新开始 schema freeze 与 parity；不得从本轮得出 posterior superiority。